In [1]:
#healthy vs rest

# ============================================================
#
# UPDATED FULL NOTEBOOK CELL (Stage 1: healthy vs rest)
# Input: plant blobs / tiles
#
# Label mapping:
#   0 = healthy
#   1 = moderately_healthy OR moderately_unhealthy OR unhealthy OR severely_unhealthy
#
# Changes requested:
#   - Keep overall logic the same
#   - Use ImageNet pretrained models only
#   - No data augmentation
#   - Train on your dataset using ImageNet initialized weights
#
# ============================================================

# -------- Imports (all used in this cell) --------
import json
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

try:
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

# -------- Paths --------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE_NAME = "stage1_healthy_vs_rest_imagenet_only_no_aug"

# -------- Original class folders --------
MULTI_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

# -------- Binary mapping --------
NEGATIVE_CLASS_NAMES = {"healthy"}
BIN_CLASS_NAMES = ["healthy", "rest"]

# -------- Small helpers --------
def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# -------- Dataset --------
class BinaryTileFolderDataset(Dataset):
    def __init__(self, tiles_root: Path, split: str, transform=None, allowed_classes=None):
        self.tiles_root = Path(tiles_root)
        self.split = split
        self.transform = transform
        self.allowed_classes = allowed_classes

        split_dir = self.tiles_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {split_dir}")

        items: List[Tuple[Path, int]] = []
        for cls_name in MULTI_CLASS_NAMES:
            if self.allowed_classes is not None and cls_name not in self.allowed_classes:
                continue

            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                print(f"WARNING: class folder missing: {cls_dir}")
                continue

            for img_path in sorted(cls_dir.rglob("*")):
                if img_path.is_file() and is_image_file(img_path):
                    y = 0 if cls_name in NEGATIVE_CLASS_NAMES else 1
                    items.append((img_path, y))

        if len(items) == 0:
            raise RuntimeError(f"No images found for split '{split}' under: {split_dir}")

        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, label = self.items[idx]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            if self.transform is not None:
                im = self.transform(im)
        return im, label

# -------- Imbalance helpers --------
def get_class_counts_binary(dataset: BinaryTileFolderDataset) -> np.ndarray:
    counts = np.zeros(2, dtype=np.int64)
    for _, y in dataset.items:
        counts[y] += 1
    return counts

def compute_class_weights_softened_binary(
    dataset: BinaryTileFolderDataset,
    weight_power: float = 0.5,
    max_weight: float = 20.0
) -> torch.Tensor:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    total = float(counts.sum())
    base = total / counts.astype(np.float32)
    softened = np.power(base, weight_power)
    softened = np.minimum(softened, max_weight)
    return torch.tensor(softened, dtype=torch.float32)

def make_weighted_sampler_binary(dataset: BinaryTileFolderDataset) -> WeightedRandomSampler:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    inv = 1.0 / counts.astype(np.float32)
    sample_weights = torch.tensor([inv[y] for _, y in dataset.items], dtype=torch.double)
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# -------- Model builder --------
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower().strip()

    if model_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet34":
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        model = resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model

    raise ValueError("Unknown model. Use resnet18, resnet34, resnet50, efficientnet_b0")

def maybe_freeze_backbone(model: nn.Module, model_name: str, freeze: bool):
    if not freeze:
        return
    model_name = model_name.lower().strip()
    if model_name.startswith("resnet"):
        for name, p in model.named_parameters():
            if not name.startswith("fc."):
                p.requires_grad = False
    elif model_name.startswith("efficientnet"):
        for name, p in model.named_parameters():
            if not name.startswith("classifier."):
                p.requires_grad = False

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

# -------- Train + eval loops --------
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accs = [], []
    all_y, all_p = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, y))

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.detach().cpu().numpy())
        all_p.append(preds.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)

    per_class_acc = np.zeros(2, dtype=np.float32)
    for k in range(2):
        mask = (y_true == k)
        per_class_acc[k] = (y_pred[mask] == k).mean() if mask.sum() > 0 else np.nan

    return float(np.mean(losses)), float(np.mean(accs)), y_true, y_pred, per_class_acc

def train_one_epoch(model, loader, optimizer, criterion, device, scheduler, scaler, grad_clip: float):
    model.train()
    losses, accs = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            if grad_clip and grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits.detach(), y))

    return float(np.mean(losses)), float(np.mean(accs))

# ============================================================
# CONFIG
# ============================================================

MODEL_NAMES = ["resnet18", "resnet34", "resnet50", "efficientnet_b0"]
EPOCHS = 20
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 4
SEED = 42

USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
WEIGHT_POWER = 0.5
MAX_WEIGHT = 20.0
LABEL_SMOOTHING = 0.0   # no extra regularization
PRETRAINED = True       # ImageNet pretrained weights
FREEZE_EPOCHS = 2       # train classifier head first, then full model
GRAD_CLIP = 1.0

SCHEDULER = "onecycle"
EARLY_STOP_PATIENCE = 6
SAVE_BEST_BY = "macro_f1"

PRINT_EVERY = 1
REPORT_EVERY = 5

# ============================================================
# SETUP
# ============================================================

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# NO AUGMENTATION
# Only ImageNet normalization is used
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_ds = BinaryTileFolderDataset(TILES_ROOT, "train", transform=train_tf, allowed_classes=None)
val_ds = BinaryTileFolderDataset(TILES_ROOT, "val", transform=val_tf, allowed_classes=None)

train_counts = get_class_counts_binary(train_ds)
print(f"Train counts (0={BIN_CLASS_NAMES[0]}, 1={BIN_CLASS_NAMES[1]}):", train_counts.tolist())

if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights_softened_binary(
        train_ds,
        weight_power=WEIGHT_POWER,
        max_weight=MAX_WEIGHT
    )
else:
    class_weights = torch.ones(2, dtype=torch.float32)

if USE_WEIGHTED_SAMPLER:
    sampler = make_weighted_sampler_binary(train_ds)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=shuffle,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

def make_scheduler(optimizer, steps_per_epoch: int):
    if SCHEDULER.lower() == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=LR,
            epochs=EPOCHS,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.2,
        )
    if SCHEDULER.lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    return None

all_results = []

# ============================================================
# TRAIN ALL MODELS
# ============================================================

for MODEL_NAME in MODEL_NAMES:
    print("\n" + "=" * 70)
    print(f"Training model: {MODEL_NAME}")
    print("=" * 70)

    set_seed(SEED)

    STAGE_OUT = OUT_ROOT / f"{STAGE_NAME}_{MODEL_NAME}"
    STAGE_OUT.mkdir(parents=True, exist_ok=True)

    model = build_model(MODEL_NAME, num_classes=2, pretrained=PRETRAINED).to(device)

    if FREEZE_EPOCHS > 0:
        maybe_freeze_backbone(model, MODEL_NAME, freeze=True)
        print(f"Backbone frozen for first {FREEZE_EPOCHS} epoch(s).")

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=float(LABEL_SMOOTHING)
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))

    best_score = -1.0
    no_improve = 0
    history: List[Dict] = []
    best_val_acc = float("nan")
    best_macro_f1 = float("nan")
    best_epoch = -1

    # ============================================================
    # TRAIN
    # ============================================================

    for epoch in range(1, EPOCHS + 1):

        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))
            scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
            print("Backbone unfrozen. Training full model now.")

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scheduler=scheduler if SCHEDULER.lower() == "onecycle" else None,
            scaler=scaler,
            grad_clip=GRAD_CLIP,
        )

        if scheduler is not None and SCHEDULER.lower() == "cosine":
            scheduler.step()

        val_loss, val_acc, y_true, y_pred, per_class_acc = evaluate(model, val_loader, criterion, device)

        if SKLEARN_OK:
            try:
                macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            except Exception as e:
                print("ERROR computing f1_score:", repr(e))
                macro_f1 = float("nan")
        else:
            macro_f1 = float("nan")

        lr_now = float(optimizer.param_groups[0]["lr"])

        history.append({
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "macro_f1": macro_f1,
            "lr": lr_now,
        })

        if epoch % PRINT_EVERY == 0:
            print(
                f"[{MODEL_NAME}] Epoch {epoch:03d}/{EPOCHS} | "
                f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
                f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} | "
                f"macro_f1 {macro_f1:.4f} | "
                f"lr {lr_now:.2e}"
            )
            print(
                f"Per-class acc: "
                f"0({BIN_CLASS_NAMES[0]})={per_class_acc[0]*100:.2f}%  "
                f"1({BIN_CLASS_NAMES[1]})={per_class_acc[1]*100:.2f}%"
            )

        if SKLEARN_OK and (epoch % REPORT_EVERY == 0 or epoch == 1):
            print(classification_report(
                y_true,
                y_pred,
                target_names=BIN_CLASS_NAMES,
                digits=4,
                zero_division=0
            ))

        score = val_acc if SAVE_BEST_BY.lower() == "val_acc" else (macro_f1 if SKLEARN_OK else val_acc)

        if score > best_score:
            best_score = score
            best_val_acc = val_acc
            best_macro_f1 = macro_f1
            best_epoch = epoch
            no_improve = 0

            torch.save({
                "model_state": model.state_dict(),
                "model": MODEL_NAME,
                "img_size": IMG_SIZE,
                "epoch": epoch,
                "val_acc": val_acc,
                "macro_f1": macro_f1,
                "bin_class_names": BIN_CLASS_NAMES,
                "negative_class_names": sorted(list(NEGATIVE_CLASS_NAMES)),
                "args": {
                    "MODEL_NAME": MODEL_NAME,
                    "EPOCHS": EPOCHS,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "IMG_SIZE": IMG_SIZE,
                    "USE_WEIGHTED_SAMPLER": USE_WEIGHTED_SAMPLER,
                    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
                    "WEIGHT_POWER": WEIGHT_POWER,
                    "MAX_WEIGHT": MAX_WEIGHT,
                    "LABEL_SMOOTHING": LABEL_SMOOTHING,
                    "PRETRAINED": PRETRAINED,
                    "FREEZE_EPOCHS": FREEZE_EPOCHS,
                    "SCHEDULER": SCHEDULER,
                    "SAVE_BEST_BY": SAVE_BEST_BY,
                    "AUGMENTATION": "NONE",
                    "INPUT_SOURCE": "plant blobs",
                    "PRETRAIN_SOURCE": "ImageNet",
                },
            }, STAGE_OUT / "best_model.pt")

            if SKLEARN_OK:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                np.save(STAGE_OUT / "confusion_matrix.npy", cm.astype(np.int32))
                rep = classification_report(
                    y_true,
                    y_pred,
                    target_names=BIN_CLASS_NAMES,
                    digits=4,
                    zero_division=0
                )
                (STAGE_OUT / "classification_report.txt").write_text(rep)

            print("Saved best model to:", STAGE_OUT / "best_model.pt")
        else:
            no_improve += 1

        if EARLY_STOP_PATIENCE > 0 and no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    (STAGE_OUT / "history.json").write_text(json.dumps(history, indent=2))
    print("Done. Outputs in:", STAGE_OUT)

    all_results.append({
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "best_macro_f1": best_macro_f1,
        "output_dir": str(STAGE_OUT),
    })

# ============================================================
# FINAL COMPARISON
# ============================================================

comparison_path = OUT_ROOT / f"{STAGE_NAME}_all_models_summary.json"
comparison_path.write_text(json.dumps(all_results, indent=2))

print("\nFinal summary:")
for row in all_results:
    print(
        f"{row['model_name']:15s} | "
        f"best_epoch={row['best_epoch']:3d} | "
        f"best_val_acc={row['best_val_acc']:.4f} | "
        f"best_macro_f1={row['best_macro_f1']:.4f}"
    )

print("\nSaved model comparison to:", comparison_path)

Device: cuda
Train counts (0=healthy, 1=rest): [11600, 18662]

Training model: resnet18
Backbone frozen for first 2 epoch(s).
[resnet18] Epoch 001/20 | train_loss 0.6846 train_acc 0.5828 | val_loss 0.6682 val_acc 0.6480 | macro_f1 0.6456 | lr 5.42e-05
Per-class acc: 0(healthy)=81.54%  1(rest)=51.18%
              precision    recall  f1-score   support

     healthy     0.5766    0.8154    0.6755      1934
        rest     0.7728    0.5118    0.6158      2372

    accuracy                         0.6482      4306
   macro avg     0.6747    0.6636    0.6456      4306
weighted avg     0.6847    0.6482    0.6426      4306

Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_healthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 002/20 | train_loss 0.6119 train_acc 0.6882 | val_loss 0.6255 val_acc 0.6886 | macro_f1 0.6868 | lr 1.56e-04
Per-class acc: 0(healthy)=85.68%  1(rest)=55.19%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_r

/tmp/ipykernel_3038612/3512138798.py:252: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet34] Epoch 003/20 | train_loss 0.2832 train_acc 0.8664 | val_loss 7.4606 val_acc 0.6234 | macro_f1 0.6072 | lr 5.42e-05
Per-class acc: 0(healthy)=90.95%  1(rest)=38.79%
[resnet34] Epoch 004/20 | train_loss 0.2296 train_acc 0.8969 | val_loss 2.4392 val_acc 0.6500 | macro_f1 0.6089 | lr 1.56e-04
Per-class acc: 0(healthy)=36.66%  1(rest)=87.90%
[resnet34] Epoch 005/20 | train_loss 0.2258 train_acc 0.8954 | val_loss 46.0667 val_acc 0.5822 | macro_f1 0.5536 | lr 2.58e-04
Per-class acc: 0(healthy)=94.11%  1(rest)=29.30%
              precision    recall  f1-score   support

     healthy     0.5204    0.9411    0.6702      1934
        rest     0.8591    0.2930    0.4370      2372

    accuracy                         0.5841      4306
   macro avg     0.6898    0.6170    0.5536      4306
weighted avg     0.7070    0.5841    0.5417      4306

[resnet34] Epoch 006/20 | train_loss 0.2166 train_acc 0.9031 | val_loss 18.1246 val_acc 0.6002 | macro_f1 0.5871 | lr 3.00e-04
Per-class acc: 0(hea

/tmp/ipykernel_3038612/3512138798.py:252: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet50] Epoch 003/20 | train_loss 0.3062 train_acc 0.8552 | val_loss 0.3292 val_acc 0.8574 | macro_f1 0.8549 | lr 5.42e-05
Per-class acc: 0(healthy)=82.26%  1(rest)=88.49%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_healthy_vs_rest_imagenet_only_no_aug_resnet50/best_model.pt
[resnet50] Epoch 004/20 | train_loss 0.2399 train_acc 0.8927 | val_loss 1.6876 val_acc 0.6000 | macro_f1 0.5819 | lr 1.56e-04
Per-class acc: 0(healthy)=91.37%  1(rest)=34.78%
[resnet50] Epoch 005/20 | train_loss 0.2233 train_acc 0.9039 | val_loss 7.2889 val_acc 0.5972 | macro_f1 0.5920 | lr 2.58e-04
Per-class acc: 0(healthy)=81.44%  1(rest)=42.37%
              precision    recall  f1-score   support

     healthy     0.5354    0.8144    0.6460      1934
        rest     0.7368    0.4237    0.5380      2372

    accuracy                         0.5992      4306
   macro avg     0.6361    0.6190    0.5920      4306
weighted avg     0.6463    0.5992    0.5865      4306

[resnet50] Epo

/tmp/ipykernel_3038612/3512138798.py:252: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[efficientnet_b0] Epoch 003/20 | train_loss 0.3322 train_acc 0.8360 | val_loss 0.2685 val_acc 0.8690 | macro_f1 0.8684 | lr 5.42e-05
Per-class acc: 0(healthy)=90.38%  1(rest)=84.02%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_healthy_vs_rest_imagenet_only_no_aug_efficientnet_b0/best_model.pt
[efficientnet_b0] Epoch 004/20 | train_loss 0.2509 train_acc 0.8860 | val_loss 1.2612 val_acc 0.6488 | macro_f1 0.6125 | lr 1.56e-04
Per-class acc: 0(healthy)=38.57%  1(rest)=86.13%
[efficientnet_b0] Epoch 005/20 | train_loss 0.2269 train_acc 0.8994 | val_loss 1.7766 val_acc 0.6523 | macro_f1 0.6246 | lr 2.58e-04
Per-class acc: 0(healthy)=42.86%  1(rest)=83.26%
              precision    recall  f1-score   support

     healthy     0.6762    0.4286    0.5247      1934
        rest     0.6412    0.8326    0.7245      2372

    accuracy                         0.6512      4306
   macro avg     0.6587    0.6306    0.6246      4306
weighted avg     0.6569    0.6512    0.63

mod_health vs rest

In [2]:
# ============================================================
#
# UPDATED FULL NOTEBOOK CELL (Stage 1: moderately healthy vs rest)
# Input: plant blobs / tiles
#
# Label mapping:
#   0 = moderately_healthy
#   1 = healthy OR moderately_unhealthy OR unhealthy OR severely_unhealthy
#
# Changes requested:
#   - Keep overall logic the same
#   - Use ImageNet pretrained models only
#   - No data augmentation
#   - Train on your dataset using ImageNet initialized weights
#
# ============================================================

# -------- Imports (all used in this cell) --------
import json
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

try:
    import sklearn
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    SKLEARN_OK = True
    print("sklearn version:", sklearn.__version__)
except Exception as e:
    SKLEARN_OK = False
    print("ERROR importing sklearn.metrics:", repr(e))

# -------- Paths --------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE_NAME = "stage1_modhealthy_vs_rest_imagenet_only_no_aug"

# -------- Original class folders --------
MULTI_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

# -------- Binary mapping --------
NEGATIVE_CLASS_NAMES = {"moderately_healthy"}
BIN_CLASS_NAMES = ["moderately_healthy", "rest"]

# -------- Small helpers --------
def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# -------- Dataset --------
class BinaryTileFolderDataset(Dataset):
    def __init__(self, tiles_root: Path, split: str, transform=None, allowed_classes=None):
        self.tiles_root = Path(tiles_root)
        self.split = split
        self.transform = transform
        self.allowed_classes = allowed_classes

        split_dir = self.tiles_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {split_dir}")

        items: List[Tuple[Path, int]] = []
        for cls_name in MULTI_CLASS_NAMES:
            if self.allowed_classes is not None and cls_name not in self.allowed_classes:
                continue

            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                print(f"WARNING: class folder missing: {cls_dir}")
                continue

            for img_path in sorted(cls_dir.rglob("*")):
                if img_path.is_file() and is_image_file(img_path):
                    y = 0 if cls_name in NEGATIVE_CLASS_NAMES else 1
                    items.append((img_path, y))

        if len(items) == 0:
            raise RuntimeError(f"No images found for split '{split}' under: {split_dir}")

        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, label = self.items[idx]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            if self.transform is not None:
                im = self.transform(im)
        return im, label

# -------- Imbalance helpers --------
def get_class_counts_binary(dataset: BinaryTileFolderDataset) -> np.ndarray:
    counts = np.zeros(2, dtype=np.int64)
    for _, y in dataset.items:
        counts[y] += 1
    return counts

def compute_class_weights_softened_binary(
    dataset: BinaryTileFolderDataset,
    weight_power: float = 0.5,
    max_weight: float = 20.0
) -> torch.Tensor:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    total = float(counts.sum())
    base = total / counts.astype(np.float32)
    softened = np.power(base, weight_power)
    softened = np.minimum(softened, max_weight)
    return torch.tensor(softened, dtype=torch.float32)

def make_weighted_sampler_binary(dataset: BinaryTileFolderDataset) -> WeightedRandomSampler:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    inv = 1.0 / counts.astype(np.float32)
    sample_weights = torch.tensor([inv[y] for _, y in dataset.items], dtype=torch.double)
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# -------- Model builder --------
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower().strip()

    if model_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet34":
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        model = resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model

    raise ValueError("Unknown model. Use resnet18, resnet34, resnet50, efficientnet_b0")

def maybe_freeze_backbone(model: nn.Module, model_name: str, freeze: bool):
    if not freeze:
        return
    model_name = model_name.lower().strip()
    if model_name.startswith("resnet"):
        for name, p in model.named_parameters():
            if not name.startswith("fc."):
                p.requires_grad = False
    elif model_name.startswith("efficientnet"):
        for name, p in model.named_parameters():
            if not name.startswith("classifier."):
                p.requires_grad = False

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

# -------- Train + eval loops --------
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accs = [], []
    all_y, all_p = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, y))

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.detach().cpu().numpy())
        all_p.append(preds.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)

    per_class_acc = np.zeros(2, dtype=np.float32)
    for k in range(2):
        mask = (y_true == k)
        per_class_acc[k] = (y_pred[mask] == k).mean() if mask.sum() > 0 else np.nan

    return float(np.mean(losses)), float(np.mean(accs)), y_true, y_pred, per_class_acc

def train_one_epoch(model, loader, optimizer, criterion, device, scheduler, scaler, grad_clip: float):
    model.train()
    losses, accs = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            if grad_clip and grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits.detach(), y))

    return float(np.mean(losses)), float(np.mean(accs))

# ============================================================
# CONFIG
# ============================================================

MODEL_NAMES = ["resnet18", "resnet34", "resnet50", "efficientnet_b0"]
EPOCHS = 20
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 4
SEED = 42

USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
WEIGHT_POWER = 0.5
MAX_WEIGHT = 20.0
LABEL_SMOOTHING = 0.0   # no extra regularization
PRETRAINED = True       # ImageNet pretrained weights
FREEZE_EPOCHS = 2
GRAD_CLIP = 1.0

SCHEDULER = "onecycle"
EARLY_STOP_PATIENCE = 6
SAVE_BEST_BY = "macro_f1"

PRINT_EVERY = 1
REPORT_EVERY = 5

# ============================================================
# SETUP
# ============================================================

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# NO AUGMENTATION
# Only ImageNet resize and normalization
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_ds = BinaryTileFolderDataset(TILES_ROOT, "train", transform=train_tf, allowed_classes=None)
val_ds = BinaryTileFolderDataset(TILES_ROOT, "val", transform=val_tf, allowed_classes=None)

train_counts = get_class_counts_binary(train_ds)
print(f"Train counts (0={BIN_CLASS_NAMES[0]}, 1={BIN_CLASS_NAMES[1]}):", train_counts.tolist())

if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights_softened_binary(
        train_ds,
        weight_power=WEIGHT_POWER,
        max_weight=MAX_WEIGHT
    )
else:
    class_weights = torch.ones(2, dtype=torch.float32)

if USE_WEIGHTED_SAMPLER:
    sampler = make_weighted_sampler_binary(train_ds)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=shuffle,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

def make_scheduler(optimizer, steps_per_epoch: int):
    if SCHEDULER.lower() == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=LR,
            epochs=EPOCHS,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.2,
        )
    if SCHEDULER.lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    return None

all_results = []

# ============================================================
# TRAIN ALL MODELS
# ============================================================

for MODEL_NAME in MODEL_NAMES:
    print("\n" + "=" * 70)
    print(f"Training model: {MODEL_NAME}")
    print("=" * 70)

    set_seed(SEED)

    STAGE_OUT = OUT_ROOT / f"{STAGE_NAME}_{MODEL_NAME}"
    STAGE_OUT.mkdir(parents=True, exist_ok=True)

    model = build_model(MODEL_NAME, num_classes=2, pretrained=PRETRAINED).to(device)

    if FREEZE_EPOCHS > 0:
        maybe_freeze_backbone(model, MODEL_NAME, freeze=True)
        print(f"Backbone frozen for first {FREEZE_EPOCHS} epoch(s).")

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=float(LABEL_SMOOTHING)
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))

    best_score = -1.0
    no_improve = 0
    history: List[Dict] = []
    best_val_acc = float("nan")
    best_macro_f1 = float("nan")
    best_epoch = -1

    # ============================================================
    # TRAIN
    # ============================================================

    for epoch in range(1, EPOCHS + 1):

        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))
            scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
            print("Backbone unfrozen. Training full model now.")

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scheduler=scheduler if SCHEDULER.lower() == "onecycle" else None,
            scaler=scaler,
            grad_clip=GRAD_CLIP,
        )

        if scheduler is not None and SCHEDULER.lower() == "cosine":
            scheduler.step()

        val_loss, val_acc, y_true, y_pred, per_class_acc = evaluate(model, val_loader, criterion, device)

        if SKLEARN_OK:
            try:
                macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            except Exception as e:
                print("ERROR computing f1_score:", repr(e))
                macro_f1 = float("nan")
        else:
            macro_f1 = float("nan")

        lr_now = float(optimizer.param_groups[0]["lr"])

        history.append({
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "macro_f1": macro_f1,
            "lr": lr_now,
        })

        if epoch % PRINT_EVERY == 0:
            print(
                f"[{MODEL_NAME}] Epoch {epoch:03d}/{EPOCHS} | "
                f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
                f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} | "
                f"macro_f1 {macro_f1:.4f} | "
                f"lr {lr_now:.2e}"
            )
            print(
                f"Per-class acc: "
                f"0({BIN_CLASS_NAMES[0]})={per_class_acc[0]*100:.2f}%  "
                f"1({BIN_CLASS_NAMES[1]})={per_class_acc[1]*100:.2f}%"
            )

        if SKLEARN_OK and (epoch % REPORT_EVERY == 0 or epoch == 1):
            print(classification_report(
                y_true,
                y_pred,
                target_names=BIN_CLASS_NAMES,
                digits=4,
                zero_division=0
            ))

        score = val_acc if SAVE_BEST_BY.lower() == "val_acc" else (macro_f1 if SKLEARN_OK else val_acc)

        if score > best_score:
            best_score = score
            best_val_acc = val_acc
            best_macro_f1 = macro_f1
            best_epoch = epoch
            no_improve = 0

            torch.save({
                "model_state": model.state_dict(),
                "model": MODEL_NAME,
                "img_size": IMG_SIZE,
                "epoch": epoch,
                "val_acc": val_acc,
                "macro_f1": macro_f1,
                "bin_class_names": BIN_CLASS_NAMES,
                "negative_class_names": sorted(list(NEGATIVE_CLASS_NAMES)),
                "args": {
                    "MODEL_NAME": MODEL_NAME,
                    "EPOCHS": EPOCHS,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "IMG_SIZE": IMG_SIZE,
                    "USE_WEIGHTED_SAMPLER": USE_WEIGHTED_SAMPLER,
                    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
                    "WEIGHT_POWER": WEIGHT_POWER,
                    "MAX_WEIGHT": MAX_WEIGHT,
                    "LABEL_SMOOTHING": LABEL_SMOOTHING,
                    "PRETRAINED": PRETRAINED,
                    "FREEZE_EPOCHS": FREEZE_EPOCHS,
                    "SCHEDULER": SCHEDULER,
                    "SAVE_BEST_BY": SAVE_BEST_BY,
                    "AUGMENTATION": "NONE",
                    "INPUT_SOURCE": "plant blobs",
                    "PRETRAIN_SOURCE": "ImageNet",
                },
            }, STAGE_OUT / "best_model.pt")

            if SKLEARN_OK:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                np.save(STAGE_OUT / "confusion_matrix.npy", cm.astype(np.int32))
                rep = classification_report(
                    y_true,
                    y_pred,
                    target_names=BIN_CLASS_NAMES,
                    digits=4,
                    zero_division=0
                )
                (STAGE_OUT / "classification_report.txt").write_text(rep)

            print("Saved best model to:", STAGE_OUT / "best_model.pt")
        else:
            no_improve += 1

        if EARLY_STOP_PATIENCE > 0 and no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    (STAGE_OUT / "history.json").write_text(json.dumps(history, indent=2))
    print("Done. Outputs in:", STAGE_OUT)

    all_results.append({
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "best_macro_f1": best_macro_f1,
        "output_dir": str(STAGE_OUT),
    })

# ============================================================
# FINAL COMPARISON
# ============================================================

comparison_path = OUT_ROOT / f"{STAGE_NAME}_all_models_summary.json"
comparison_path.write_text(json.dumps(all_results, indent=2))

print("\nFinal summary:")
for row in all_results:
    print(
        f"{row['model_name']:15s} | "
        f"best_epoch={row['best_epoch']:3d} | "
        f"best_val_acc={row['best_val_acc']:.4f} | "
        f"best_macro_f1={row['best_macro_f1']:.4f}"
    )

print("\nSaved model comparison to:", comparison_path)

sklearn version: 1.8.0
Device: cuda
Train counts (0=moderately_healthy, 1=rest): [10782, 19480]

Training model: resnet18
Backbone frozen for first 2 epoch(s).
[resnet18] Epoch 001/20 | train_loss 0.7098 train_acc 0.5066 | val_loss 0.7991 val_acc 0.3569 | macro_f1 0.3260 | lr 5.42e-05
Per-class acc: 0(moderately_healthy)=90.31%  1(rest)=10.29%
                    precision    recall  f1-score   support

moderately_healthy     0.3201    0.9031    0.4726      1372
              rest     0.6943    0.1029    0.1793      2934

          accuracy                         0.3579      4306
         macro avg     0.5072    0.5030    0.3260      4306
      weighted avg     0.5750    0.3579    0.2727      4306

Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt


[resnet18] Epoch 002/20 | train_loss 0.6911 train_acc 0.5173 | val_loss 0.7762 val_acc 0.3961 | macro_f1 0.3838 | lr 1.56e-04
Per-class acc: 0(moderately_healthy)=85.64%  1(rest)=18.27%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
Backbone unfrozen. Training full model now.


/tmp/ipykernel_3038612/3778614720.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet18] Epoch 003/20 | train_loss 0.4932 train_acc 0.7055 | val_loss 1.5957 val_acc 0.4384 | macro_f1 0.4324 | lr 5.42e-05
Per-class acc: 0(moderately_healthy)=55.03%  1(rest)=38.34%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 004/20 | train_loss 0.3928 train_acc 0.7840 | val_loss 2.2863 val_acc 0.5123 | macro_f1 0.4667 | lr 1.56e-04
Per-class acc: 0(moderately_healthy)=35.06%  1(rest)=58.55%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 005/20 | train_loss 0.3551 train_acc 0.8102 | val_loss 8.4438 val_acc 0.6025 | macro_f1 0.5519 | lr 2.58e-04
Per-class acc: 0(moderately_healthy)=42.27%  1(rest)=68.47%
                    precision    recall  f1-score   support

moderately_healthy     0.3854    0.4227    0.4032      1372
              rest     0.7172    0.6847    0.70

/tmp/ipykernel_3038612/3778614720.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet34] Epoch 003/20 | train_loss 0.4379 train_acc 0.7482 | val_loss 2.3902 val_acc 0.6236 | macro_f1 0.5337 | lr 5.42e-05
Per-class acc: 0(moderately_healthy)=29.23%  1(rest)=77.68%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modhealthy_vs_rest_imagenet_only_no_aug_resnet34/best_model.pt
[resnet34] Epoch 004/20 | train_loss 0.3673 train_acc 0.8058 | val_loss 3.2898 val_acc 0.4164 | macro_f1 0.3991 | lr 1.56e-04
Per-class acc: 0(moderately_healthy)=93.22%  1(rest)=17.72%
[resnet34] Epoch 005/20 | train_loss 0.3422 train_acc 0.8198 | val_loss 0.6044 val_acc 0.6192 | macro_f1 0.5614 | lr 2.58e-04
Per-class acc: 0(moderately_healthy)=40.60%  1(rest)=71.71%
                    precision    recall  f1-score   support

moderately_healthy     0.4016    0.4060    0.4038      1372
              rest     0.7208    0.7171    0.7189      2934

          accuracy                         0.6180      4306
         macro avg     0.5612    0.5615    0.5614      4306
   

mod_unhealthy vs rest

In [3]:
# ============================================================
#
# UPDATED FULL NOTEBOOK CELL (Stage 1: moderately unhealthy vs rest)
# Input: plant blobs / tiles
#
# Label mapping:
#   0 = moderately_unhealthy
#   1 = healthy OR moderately_healthy OR unhealthy OR severely_unhealthy
#
# Changes requested:
#   - Keep overall logic the same
#   - Use ImageNet pretrained models only
#   - No data augmentation
#   - Train on your dataset using ImageNet initialized weights
#
# ============================================================

# -------- Imports (all used in this cell) --------
import json
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

try:
    import sklearn
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    SKLEARN_OK = True
    print("sklearn version:", sklearn.__version__)
except Exception as e:
    SKLEARN_OK = False
    print("ERROR importing sklearn.metrics:", repr(e))

# -------- Paths --------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE_NAME = "stage1_modunhealthy_vs_rest_imagenet_only_no_aug"

# -------- Original class folders --------
MULTI_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

# -------- Binary mapping --------
NEGATIVE_CLASS_NAMES = {"moderately_unhealthy"}
BIN_CLASS_NAMES = ["moderately_unhealthy", "rest"]

# -------- Small helpers --------
def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# -------- Dataset --------
class BinaryTileFolderDataset(Dataset):
    def __init__(self, tiles_root: Path, split: str, transform=None, allowed_classes=None):
        self.tiles_root = Path(tiles_root)
        self.split = split
        self.transform = transform
        self.allowed_classes = allowed_classes

        split_dir = self.tiles_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {split_dir}")

        items: List[Tuple[Path, int]] = []
        for cls_name in MULTI_CLASS_NAMES:
            if self.allowed_classes is not None and cls_name not in self.allowed_classes:
                continue

            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                print(f"WARNING: class folder missing: {cls_dir}")
                continue

            for img_path in sorted(cls_dir.rglob("*")):
                if img_path.is_file() and is_image_file(img_path):
                    y = 0 if cls_name in NEGATIVE_CLASS_NAMES else 1
                    items.append((img_path, y))

        if len(items) == 0:
            raise RuntimeError(f"No images found for split '{split}' under: {split_dir}")

        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, label = self.items[idx]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            if self.transform is not None:
                im = self.transform(im)
        return im, label

# -------- Imbalance helpers --------
def get_class_counts_binary(dataset: BinaryTileFolderDataset) -> np.ndarray:
    counts = np.zeros(2, dtype=np.int64)
    for _, y in dataset.items:
        counts[y] += 1
    return counts

def compute_class_weights_softened_binary(
    dataset: BinaryTileFolderDataset,
    weight_power: float = 0.5,
    max_weight: float = 20.0
) -> torch.Tensor:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    total = float(counts.sum())
    base = total / counts.astype(np.float32)
    softened = np.power(base, weight_power)
    softened = np.minimum(softened, max_weight)
    return torch.tensor(softened, dtype=torch.float32)

def make_weighted_sampler_binary(dataset: BinaryTileFolderDataset) -> WeightedRandomSampler:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    inv = 1.0 / counts.astype(np.float32)
    sample_weights = torch.tensor([inv[y] for _, y in dataset.items], dtype=torch.double)
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# -------- Model builder --------
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower().strip()

    if model_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet34":
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        model = resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model

    raise ValueError("Unknown model. Use resnet18, resnet34, resnet50, efficientnet_b0")

def maybe_freeze_backbone(model: nn.Module, model_name: str, freeze: bool):
    if not freeze:
        return
    model_name = model_name.lower().strip()
    if model_name.startswith("resnet"):
        for name, p in model.named_parameters():
            if not name.startswith("fc."):
                p.requires_grad = False
    elif model_name.startswith("efficientnet"):
        for name, p in model.named_parameters():
            if not name.startswith("classifier."):
                p.requires_grad = False

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

# -------- Train + eval loops --------
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accs = [], []
    all_y, all_p = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, y))

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.detach().cpu().numpy())
        all_p.append(preds.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)

    per_class_acc = np.zeros(2, dtype=np.float32)
    for k in range(2):
        mask = (y_true == k)
        per_class_acc[k] = (y_pred[mask] == k).mean() if mask.sum() > 0 else np.nan

    return float(np.mean(losses)), float(np.mean(accs)), y_true, y_pred, per_class_acc

def train_one_epoch(model, loader, optimizer, criterion, device, scheduler, scaler, grad_clip: float):
    model.train()
    losses, accs = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            if grad_clip and grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits.detach(), y))

    return float(np.mean(losses)), float(np.mean(accs))

# ============================================================
# CONFIG
# ============================================================

MODEL_NAMES = ["resnet18", "resnet34", "resnet50", "efficientnet_b0"]
EPOCHS = 20
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 4
SEED = 42

USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
WEIGHT_POWER = 0.5
MAX_WEIGHT = 20.0
LABEL_SMOOTHING = 0.0   # no extra regularization
PRETRAINED = True       # ImageNet pretrained weights
FREEZE_EPOCHS = 2
GRAD_CLIP = 1.0

SCHEDULER = "onecycle"
EARLY_STOP_PATIENCE = 6
SAVE_BEST_BY = "macro_f1"

PRINT_EVERY = 1
REPORT_EVERY = 5

# ============================================================
# SETUP
# ============================================================

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# NO AUGMENTATION
# Only ImageNet resize and normalization
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_ds = BinaryTileFolderDataset(TILES_ROOT, "train", transform=train_tf, allowed_classes=None)
val_ds = BinaryTileFolderDataset(TILES_ROOT, "val", transform=val_tf, allowed_classes=None)

train_counts = get_class_counts_binary(train_ds)
print(f"Train counts (0={BIN_CLASS_NAMES[0]}, 1={BIN_CLASS_NAMES[1]}):", train_counts.tolist())

if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights_softened_binary(
        train_ds,
        weight_power=WEIGHT_POWER,
        max_weight=MAX_WEIGHT
    )
else:
    class_weights = torch.ones(2, dtype=torch.float32)

if USE_WEIGHTED_SAMPLER:
    sampler = make_weighted_sampler_binary(train_ds)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=shuffle,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

def make_scheduler(optimizer, steps_per_epoch: int):
    if SCHEDULER.lower() == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=LR,
            epochs=EPOCHS,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.2,
        )
    if SCHEDULER.lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    return None

all_results = []

# ============================================================
# TRAIN ALL MODELS
# ============================================================

for MODEL_NAME in MODEL_NAMES:
    print("\n" + "=" * 70)
    print(f"Training model: {MODEL_NAME}")
    print("=" * 70)

    set_seed(SEED)

    STAGE_OUT = OUT_ROOT / f"{STAGE_NAME}_{MODEL_NAME}"
    STAGE_OUT.mkdir(parents=True, exist_ok=True)

    model = build_model(MODEL_NAME, num_classes=2, pretrained=PRETRAINED).to(device)

    if FREEZE_EPOCHS > 0:
        maybe_freeze_backbone(model, MODEL_NAME, freeze=True)
        print(f"Backbone frozen for first {FREEZE_EPOCHS} epoch(s).")

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=float(LABEL_SMOOTHING)
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))

    best_score = -1.0
    no_improve = 0
    history: List[Dict] = []
    best_val_acc = float("nan")
    best_macro_f1 = float("nan")
    best_epoch = -1

    # ============================================================
    # TRAIN
    # ============================================================

    for epoch in range(1, EPOCHS + 1):

        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))
            scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
            print("Backbone unfrozen. Training full model now.")

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scheduler=scheduler if SCHEDULER.lower() == "onecycle" else None,
            scaler=scaler,
            grad_clip=GRAD_CLIP,
        )

        if scheduler is not None and SCHEDULER.lower() == "cosine":
            scheduler.step()

        val_loss, val_acc, y_true, y_pred, per_class_acc = evaluate(model, val_loader, criterion, device)

        if SKLEARN_OK:
            try:
                macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            except Exception as e:
                print("ERROR computing f1_score:", repr(e))
                macro_f1 = float("nan")
        else:
            macro_f1 = float("nan")

        lr_now = float(optimizer.param_groups[0]["lr"])

        history.append({
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "macro_f1": macro_f1,
            "lr": lr_now,
        })

        if epoch % PRINT_EVERY == 0:
            print(
                f"[{MODEL_NAME}] Epoch {epoch:03d}/{EPOCHS} | "
                f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
                f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} | "
                f"macro_f1 {macro_f1:.4f} | "
                f"lr {lr_now:.2e}"
            )
            print(
                f"Per-class acc: "
                f"0({BIN_CLASS_NAMES[0]})={per_class_acc[0]*100:.2f}%  "
                f"1({BIN_CLASS_NAMES[1]})={per_class_acc[1]*100:.2f}%"
            )

        if SKLEARN_OK and (epoch % REPORT_EVERY == 0 or epoch == 1):
            print(classification_report(
                y_true,
                y_pred,
                target_names=BIN_CLASS_NAMES,
                digits=4,
                zero_division=0
            ))

        score = val_acc if SAVE_BEST_BY.lower() == "val_acc" else (macro_f1 if SKLEARN_OK else val_acc)

        if score > best_score:
            best_score = score
            best_val_acc = val_acc
            best_macro_f1 = macro_f1
            best_epoch = epoch
            no_improve = 0

            torch.save({
                "model_state": model.state_dict(),
                "model": MODEL_NAME,
                "img_size": IMG_SIZE,
                "epoch": epoch,
                "val_acc": val_acc,
                "macro_f1": macro_f1,
                "bin_class_names": BIN_CLASS_NAMES,
                "negative_class_names": sorted(list(NEGATIVE_CLASS_NAMES)),
                "args": {
                    "MODEL_NAME": MODEL_NAME,
                    "EPOCHS": EPOCHS,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "IMG_SIZE": IMG_SIZE,
                    "USE_WEIGHTED_SAMPLER": USE_WEIGHTED_SAMPLER,
                    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
                    "WEIGHT_POWER": WEIGHT_POWER,
                    "MAX_WEIGHT": MAX_WEIGHT,
                    "LABEL_SMOOTHING": LABEL_SMOOTHING,
                    "PRETRAINED": PRETRAINED,
                    "FREEZE_EPOCHS": FREEZE_EPOCHS,
                    "SCHEDULER": SCHEDULER,
                    "SAVE_BEST_BY": SAVE_BEST_BY,
                    "AUGMENTATION": "NONE",
                    "INPUT_SOURCE": "plant blobs",
                    "PRETRAIN_SOURCE": "ImageNet",
                },
            }, STAGE_OUT / "best_model.pt")

            if SKLEARN_OK:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                np.save(STAGE_OUT / "confusion_matrix.npy", cm.astype(np.int32))
                rep = classification_report(
                    y_true,
                    y_pred,
                    target_names=BIN_CLASS_NAMES,
                    digits=4,
                    zero_division=0
                )
                (STAGE_OUT / "classification_report.txt").write_text(rep)

            print("Saved best model to:", STAGE_OUT / "best_model.pt")
        else:
            no_improve += 1

        if EARLY_STOP_PATIENCE > 0 and no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    (STAGE_OUT / "history.json").write_text(json.dumps(history, indent=2))
    print("Done. Outputs in:", STAGE_OUT)

    all_results.append({
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "best_macro_f1": best_macro_f1,
        "output_dir": str(STAGE_OUT),
    })

# ============================================================
# FINAL COMPARISON
# ============================================================

comparison_path = OUT_ROOT / f"{STAGE_NAME}_all_models_summary.json"
comparison_path.write_text(json.dumps(all_results, indent=2))

print("\nFinal summary:")
for row in all_results:
    print(
        f"{row['model_name']:15s} | "
        f"best_epoch={row['best_epoch']:3d} | "
        f"best_val_acc={row['best_val_acc']:.4f} | "
        f"best_macro_f1={row['best_macro_f1']:.4f}"
    )

print("\nSaved model comparison to:", comparison_path)

sklearn version: 1.8.0
Device: cuda
Train counts (0=moderately_unhealthy, 1=rest): [6136, 24126]

Training model: resnet18
Backbone frozen for first 2 epoch(s).
[resnet18] Epoch 001/20 | train_loss 0.6574 train_acc 0.5038 | val_loss 0.9549 val_acc 0.2590 | macro_f1 0.2508 | lr 5.42e-05
Per-class acc: 0(moderately_unhealthy)=94.43%  1(rest)=9.30%
                      precision    recall  f1-score   support

moderately_unhealthy     0.2024    0.9443    0.3334       844
                rest     0.8726    0.0930    0.1681      3462

            accuracy                         0.2599      4306
           macro avg     0.5375    0.5187    0.2508      4306
        weighted avg     0.7413    0.2599    0.2005      4306

Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 002/20 | train_loss 0.5981 train_acc 0.5539 | val_loss 0.8249 val_acc 0.4185 | macro_f1 0.4176 | lr 1.56e-04
Per-class ac

[resnet18] Epoch 003/20 | train_loss 0.3400 train_acc 0.8178 | val_loss 2.8495 val_acc 0.3782 | macro_f1 0.3545 | lr 5.42e-05
Per-class acc: 0(moderately_unhealthy)=49.17%  1(rest)=34.81%
[resnet18] Epoch 004/20 | train_loss 0.2580 train_acc 0.8794 | val_loss 1.1684 val_acc 0.7433 | macro_f1 0.6022 | lr 1.56e-04
Per-class acc: 0(moderately_unhealthy)=37.91%  1(rest)=83.10%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 005/20 | train_loss 0.2492 train_acc 0.8861 | val_loss 2.1798 val_acc 0.7183 | macro_f1 0.5982 | lr 2.58e-04
Per-class acc: 0(moderately_unhealthy)=44.08%  1(rest)=78.48%
                      precision    recall  f1-score   support

moderately_unhealthy     0.3330    0.4408    0.3794       844
                rest     0.8520    0.7848    0.8170      3462

            accuracy                         0.7174      4306
           macro avg     0.5925    0.6128    0.

/tmp/ipykernel_3038612/3853939266.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet34] Epoch 003/20 | train_loss 0.3121 train_acc 0.8362 | val_loss 1.4579 val_acc 0.6553 | macro_f1 0.5550 | lr 5.42e-05
Per-class acc: 0(moderately_unhealthy)=46.45%  1(rest)=70.05%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_resnet34/best_model.pt
[resnet34] Epoch 004/20 | train_loss 0.2617 train_acc 0.8755 | val_loss 1.1968 val_acc 0.7819 | macro_f1 0.6279 | lr 1.56e-04
Per-class acc: 0(moderately_unhealthy)=35.55%  1(rest)=88.50%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_resnet34/best_model.pt
[resnet34] Epoch 005/20 | train_loss 0.2605 train_acc 0.8781 | val_loss 1.0454 val_acc 0.7843 | macro_f1 0.6165 | lr 2.58e-04
Per-class acc: 0(moderately_unhealthy)=31.52%  1(rest)=89.77%
                      precision    recall  f1-score   support

moderately_unhealthy     0.4290    0.3152    0.3634       844
                rest     0.8432  

/tmp/ipykernel_3038612/3853939266.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet50] Epoch 003/20 | train_loss 0.3627 train_acc 0.8015 | val_loss 0.5101 val_acc 0.7692 | macro_f1 0.6535 | lr 5.42e-05
Per-class acc: 0(moderately_unhealthy)=48.82%  1(rest)=83.77%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_resnet50/best_model.pt
[resnet50] Epoch 004/20 | train_loss 0.2426 train_acc 0.8862 | val_loss 2.2420 val_acc 0.3090 | macro_f1 0.3057 | lr 1.56e-04
Per-class acc: 0(moderately_unhealthy)=99.29%  1(rest)=14.36%
[resnet50] Epoch 005/20 | train_loss 0.2244 train_acc 0.8996 | val_loss 1.1936 val_acc 0.5727 | macro_f1 0.5010 | lr 2.58e-04
Per-class acc: 0(moderately_unhealthy)=50.00%  1(rest)=58.87%
                      precision    recall  f1-score   support

moderately_unhealthy     0.2286    0.5000    0.3138       844
                rest     0.8285    0.5887    0.6883      3462

            accuracy                         0.5713      4306
           macro avg     0.5285    0.5443    0.

/tmp/ipykernel_3038612/3853939266.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[efficientnet_b0] Epoch 003/20 | train_loss 0.3935 train_acc 0.7760 | val_loss 0.3966 val_acc 0.8179 | macro_f1 0.7712 | lr 5.42e-05
Per-class acc: 0(moderately_unhealthy)=93.84%  1(rest)=78.80%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_efficientnet_b0/best_model.pt
[efficientnet_b0] Epoch 004/20 | train_loss 0.2631 train_acc 0.8680 | val_loss 0.2788 val_acc 0.8734 | macro_f1 0.8244 | lr 1.56e-04
Per-class acc: 0(moderately_unhealthy)=88.51%  1(rest)=87.00%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_modunhealthy_vs_rest_imagenet_only_no_aug_efficientnet_b0/best_model.pt
[efficientnet_b0] Epoch 005/20 | train_loss 0.2256 train_acc 0.8969 | val_loss 0.8063 val_acc 0.7738 | macro_f1 0.6236 | lr 2.58e-04
Per-class acc: 0(moderately_unhealthy)=35.55%  1(rest)=87.90%
                      precision    recall  f1-score   support

moderately_unhealthy     0.4172    0.3555    0.3839       84

unhealthy vs rest


In [4]:
# ============================================================
#
# UPDATED FULL NOTEBOOK CELL (Stage 1: unhealthy vs rest)
# Input: plant blobs / tiles
#
# Label mapping:
#   0 = unhealthy
#   1 = healthy OR moderately_healthy OR moderately_unhealthy OR severely_unhealthy
#
# Changes requested:
#   - Keep overall logic the same
#   - Use ImageNet pretrained models only
#   - No data augmentation
#   - Train on your dataset using ImageNet initialized weights
#
# ============================================================

# -------- Imports (all used in this cell) --------
import json
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

try:
    import sklearn
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    SKLEARN_OK = True
    print("sklearn version:", sklearn.__version__)
except Exception as e:
    SKLEARN_OK = False
    print("ERROR importing sklearn.metrics:", repr(e))

# -------- Paths --------
TILES_ROOT = Path("/home/ehr/Desktop/anupama/plant/class_tiles")
OUT_ROOT = Path("/home/ehr/Desktop/anupama/plant/training_runs")
STAGE_NAME = "stage1_unhealthy_vs_rest_imagenet_only_no_aug"

# -------- Original class folders --------
MULTI_CLASS_NAMES = [
    "healthy",
    "moderately_healthy",
    "moderately_unhealthy",
    "unhealthy",
    "severely_unhealthy",
]

# -------- Binary mapping --------
NEGATIVE_CLASS_NAMES = {"unhealthy"}
BIN_CLASS_NAMES = ["unhealthy", "rest"]

# -------- Small helpers --------
def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

# -------- Dataset --------
class BinaryTileFolderDataset(Dataset):
    def __init__(self, tiles_root: Path, split: str, transform=None, allowed_classes=None):
        self.tiles_root = Path(tiles_root)
        self.split = split
        self.transform = transform
        self.allowed_classes = allowed_classes

        split_dir = self.tiles_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {split_dir}")

        items: List[Tuple[Path, int]] = []
        for cls_name in MULTI_CLASS_NAMES:
            if self.allowed_classes is not None and cls_name not in self.allowed_classes:
                continue

            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                print(f"WARNING: class folder missing: {cls_dir}")
                continue

            for img_path in sorted(cls_dir.rglob("*")):
                if img_path.is_file() and is_image_file(img_path):
                    y = 0 if cls_name in NEGATIVE_CLASS_NAMES else 1
                    items.append((img_path, y))

        if len(items) == 0:
            raise RuntimeError(f"No images found for split '{split}' under: {split_dir}")

        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, label = self.items[idx]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            if self.transform is not None:
                im = self.transform(im)
        return im, label

# -------- Imbalance helpers --------
def get_class_counts_binary(dataset: BinaryTileFolderDataset) -> np.ndarray:
    counts = np.zeros(2, dtype=np.int64)
    for _, y in dataset.items:
        counts[y] += 1
    return counts

def compute_class_weights_softened_binary(
    dataset: BinaryTileFolderDataset,
    weight_power: float = 0.5,
    max_weight: float = 20.0
) -> torch.Tensor:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    total = float(counts.sum())
    base = total / counts.astype(np.float32)
    softened = np.power(base, weight_power)
    softened = np.minimum(softened, max_weight)
    return torch.tensor(softened, dtype=torch.float32)

def make_weighted_sampler_binary(dataset: BinaryTileFolderDataset) -> WeightedRandomSampler:
    counts = np.maximum(get_class_counts_binary(dataset), 1)
    inv = 1.0 / counts.astype(np.float32)
    sample_weights = torch.tensor([inv[y] for _, y in dataset.items], dtype=torch.double)
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# -------- Model builder --------
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower().strip()

    if model_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet34":
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        model = resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if model_name == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model

    raise ValueError("Unknown model. Use resnet18, resnet34, resnet50, efficientnet_b0")

def maybe_freeze_backbone(model: nn.Module, model_name: str, freeze: bool):
    if not freeze:
        return
    model_name = model_name.lower().strip()
    if model_name.startswith("resnet"):
        for name, p in model.named_parameters():
            if not name.startswith("fc."):
                p.requires_grad = False
    elif model_name.startswith("efficientnet"):
        for name, p in model.named_parameters():
            if not name.startswith("classifier."):
                p.requires_grad = False

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

# -------- Train + eval loops --------
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accs = [], []
    all_y, all_p = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits, y))

        preds = torch.argmax(logits, dim=1)
        all_y.append(y.detach().cpu().numpy())
        all_p.append(preds.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)

    per_class_acc = np.zeros(2, dtype=np.float32)
    for k in range(2):
        mask = (y_true == k)
        per_class_acc[k] = (y_pred[mask] == k).mean() if mask.sum() > 0 else np.nan

    return float(np.mean(losses)), float(np.mean(accs)), y_true, y_pred, per_class_acc

def train_one_epoch(model, loader, optimizer, criterion, device, scheduler, scaler, grad_clip: float):
    model.train()
    losses, accs = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            if grad_clip and grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        losses.append(loss.item())
        accs.append(accuracy_from_logits(logits.detach(), y))

    return float(np.mean(losses)), float(np.mean(accs))

# ============================================================
# CONFIG
# ============================================================

MODEL_NAMES = ["resnet18", "resnet34", "resnet50", "efficientnet_b0"]
EPOCHS = 20
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 4
SEED = 42

USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
WEIGHT_POWER = 0.5
MAX_WEIGHT = 20.0
LABEL_SMOOTHING = 0.0   # no extra regularization
PRETRAINED = True       # ImageNet pretrained weights
FREEZE_EPOCHS = 2
GRAD_CLIP = 1.0

SCHEDULER = "onecycle"
EARLY_STOP_PATIENCE = 6
SAVE_BEST_BY = "macro_f1"

PRINT_EVERY = 1
REPORT_EVERY = 5

# ============================================================
# SETUP
# ============================================================

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# NO AUGMENTATION
# Only ImageNet resize and normalization
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_ds = BinaryTileFolderDataset(TILES_ROOT, "train", transform=train_tf, allowed_classes=None)
val_ds = BinaryTileFolderDataset(TILES_ROOT, "val", transform=val_tf, allowed_classes=None)

train_counts = get_class_counts_binary(train_ds)
print(f"Train counts (0={BIN_CLASS_NAMES[0]}, 1={BIN_CLASS_NAMES[1]}):", train_counts.tolist())

if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights_softened_binary(
        train_ds,
        weight_power=WEIGHT_POWER,
        max_weight=MAX_WEIGHT
    )
else:
    class_weights = torch.ones(2, dtype=torch.float32)

if USE_WEIGHTED_SAMPLER:
    sampler = make_weighted_sampler_binary(train_ds)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=shuffle,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

def make_scheduler(optimizer, steps_per_epoch: int):
    if SCHEDULER.lower() == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=LR,
            epochs=EPOCHS,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.2,
        )
    if SCHEDULER.lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    return None

all_results = []

# ============================================================
# TRAIN ALL MODELS
# ============================================================

for MODEL_NAME in MODEL_NAMES:
    print("\n" + "=" * 70)
    print(f"Training model: {MODEL_NAME}")
    print("=" * 70)

    set_seed(SEED)

    STAGE_OUT = OUT_ROOT / f"{STAGE_NAME}_{MODEL_NAME}"
    STAGE_OUT.mkdir(parents=True, exist_ok=True)

    model = build_model(MODEL_NAME, num_classes=2, pretrained=PRETRAINED).to(device)

    if FREEZE_EPOCHS > 0:
        maybe_freeze_backbone(model, MODEL_NAME, freeze=True)
        print(f"Backbone frozen for first {FREEZE_EPOCHS} epoch(s).")

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=float(LABEL_SMOOTHING)
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))

    best_score = -1.0
    no_improve = 0
    history: List[Dict] = []
    best_val_acc = float("nan")
    best_macro_f1 = float("nan")
    best_epoch = -1

    # ============================================================
    # TRAIN
    # ============================================================

    for epoch in range(1, EPOCHS + 1):

        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))
            scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
            print("Backbone unfrozen. Training full model now.")

        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scheduler=scheduler if SCHEDULER.lower() == "onecycle" else None,
            scaler=scaler,
            grad_clip=GRAD_CLIP,
        )

        if scheduler is not None and SCHEDULER.lower() == "cosine":
            scheduler.step()

        val_loss, val_acc, y_true, y_pred, per_class_acc = evaluate(model, val_loader, criterion, device)

        if SKLEARN_OK:
            try:
                macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            except Exception as e:
                print("ERROR computing f1_score:", repr(e))
                macro_f1 = float("nan")
        else:
            macro_f1 = float("nan")

        lr_now = float(optimizer.param_groups[0]["lr"])

        history.append({
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "macro_f1": macro_f1,
            "lr": lr_now,
        })

        if epoch % PRINT_EVERY == 0:
            print(
                f"[{MODEL_NAME}] Epoch {epoch:03d}/{EPOCHS} | "
                f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
                f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} | "
                f"macro_f1 {macro_f1:.4f} | "
                f"lr {lr_now:.2e}"
            )
            print(
                f"Per-class acc: "
                f"0({BIN_CLASS_NAMES[0]})={per_class_acc[0]*100:.2f}%  "
                f"1({BIN_CLASS_NAMES[1]})={per_class_acc[1]*100:.2f}%"
            )

        if SKLEARN_OK and (epoch % REPORT_EVERY == 0 or epoch == 1):
            print(classification_report(
                y_true,
                y_pred,
                target_names=BIN_CLASS_NAMES,
                digits=4,
                zero_division=0
            ))

        score = val_acc if SAVE_BEST_BY.lower() == "val_acc" else (macro_f1 if SKLEARN_OK else val_acc)

        if score > best_score:
            best_score = score
            best_val_acc = val_acc
            best_macro_f1 = macro_f1
            best_epoch = epoch
            no_improve = 0

            torch.save({
                "model_state": model.state_dict(),
                "model": MODEL_NAME,
                "img_size": IMG_SIZE,
                "epoch": epoch,
                "val_acc": val_acc,
                "macro_f1": macro_f1,
                "bin_class_names": BIN_CLASS_NAMES,
                "negative_class_names": sorted(list(NEGATIVE_CLASS_NAMES)),
                "args": {
                    "MODEL_NAME": MODEL_NAME,
                    "EPOCHS": EPOCHS,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "IMG_SIZE": IMG_SIZE,
                    "USE_WEIGHTED_SAMPLER": USE_WEIGHTED_SAMPLER,
                    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
                    "WEIGHT_POWER": WEIGHT_POWER,
                    "MAX_WEIGHT": MAX_WEIGHT,
                    "LABEL_SMOOTHING": LABEL_SMOOTHING,
                    "PRETRAINED": PRETRAINED,
                    "FREEZE_EPOCHS": FREEZE_EPOCHS,
                    "SCHEDULER": SCHEDULER,
                    "SAVE_BEST_BY": SAVE_BEST_BY,
                    "AUGMENTATION": "NONE",
                    "INPUT_SOURCE": "plant blobs",
                    "PRETRAIN_SOURCE": "ImageNet",
                },
            }, STAGE_OUT / "best_model.pt")

            if SKLEARN_OK:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                np.save(STAGE_OUT / "confusion_matrix.npy", cm.astype(np.int32))
                rep = classification_report(
                    y_true,
                    y_pred,
                    target_names=BIN_CLASS_NAMES,
                    digits=4,
                    zero_division=0
                )
                (STAGE_OUT / "classification_report.txt").write_text(rep)

            print("Saved best model to:", STAGE_OUT / "best_model.pt")
        else:
            no_improve += 1

        if EARLY_STOP_PATIENCE > 0 and no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    (STAGE_OUT / "history.json").write_text(json.dumps(history, indent=2))
    print("Done. Outputs in:", STAGE_OUT)

    all_results.append({
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "best_macro_f1": best_macro_f1,
        "output_dir": str(STAGE_OUT),
    })

# ============================================================
# FINAL COMPARISON
# ============================================================

comparison_path = OUT_ROOT / f"{STAGE_NAME}_all_models_summary.json"
comparison_path.write_text(json.dumps(all_results, indent=2))

print("\nFinal summary:")
for row in all_results:
    print(
        f"{row['model_name']:15s} | "
        f"best_epoch={row['best_epoch']:3d} | "
        f"best_val_acc={row['best_val_acc']:.4f} | "
        f"best_macro_f1={row['best_macro_f1']:.4f}"
    )

print("\nSaved model comparison to:", comparison_path)

sklearn version: 1.8.0
Device: cuda
Train counts (0=unhealthy, 1=rest): [1744, 28518]

Training model: resnet18
Backbone frozen for first 2 epoch(s).
[resnet18] Epoch 001/20 | train_loss 0.5318 train_acc 0.4956 | val_loss 1.4630 val_acc 0.0611 | macro_f1 0.0579 | lr 5.42e-05
Per-class acc: 0(unhealthy)=98.72%  1(rest)=2.31%
              precision    recall  f1-score   support

   unhealthy     0.0366    0.9872    0.0706       156
        rest     0.9796    0.0231    0.0452      4150

    accuracy                         0.0581      4306
   macro avg     0.5081    0.5052    0.0579      4306
weighted avg     0.9454    0.0581    0.0461      4306

Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 002/20 | train_loss 0.3999 train_acc 0.5597 | val_loss 0.9241 val_acc 0.4114 | macro_f1 0.3315 | lr 1.56e-04
Per-class acc: 0(unhealthy)=93.59%  1(rest)=38.99%
Saved best model to: /home/ehr/Des

[resnet18] Epoch 003/20 | train_loss 0.1227 train_acc 0.9202 | val_loss 0.8222 val_acc 0.8727 | macro_f1 0.6293 | lr 5.42e-05
Per-class acc: 0(unhealthy)=86.54%  1(rest)=87.25%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 004/20 | train_loss 0.0809 train_acc 0.9640 | val_loss 0.3754 val_acc 0.9403 | macro_f1 0.6406 | lr 1.56e-04
Per-class acc: 0(unhealthy)=35.26%  1(rest)=96.55%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_resnet18/best_model.pt
[resnet18] Epoch 005/20 | train_loss 0.0637 train_acc 0.9707 | val_loss 0.2023 val_acc 0.9236 | macro_f1 0.6326 | lr 2.58e-04
Per-class acc: 0(unhealthy)=44.23%  1(rest)=94.48%
              precision    recall  f1-score   support

   unhealthy     0.2315    0.4423    0.3040       156
        rest     0.9783    0.9448    0.9613      4150

    accuracy                     

/tmp/ipykernel_3038612/3346683839.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[resnet50] Epoch 003/20 | train_loss 0.1506 train_acc 0.8977 | val_loss 4.2218 val_acc 0.4963 | macro_f1 0.3794 | lr 5.42e-05
Per-class acc: 0(unhealthy)=87.82%  1(rest)=48.02%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_resnet50/best_model.pt
[resnet50] Epoch 004/20 | train_loss 0.0731 train_acc 0.9659 | val_loss 0.3294 val_acc 0.9190 | macro_f1 0.6204 | lr 1.56e-04
Per-class acc: 0(unhealthy)=42.31%  1(rest)=94.07%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_resnet50/best_model.pt
[resnet50] Epoch 005/20 | train_loss 0.0646 train_acc 0.9703 | val_loss 0.5881 val_acc 0.8241 | macro_f1 0.5326 | lr 2.58e-04
Per-class acc: 0(unhealthy)=46.15%  1(rest)=84.05%
              precision    recall  f1-score   support

   unhealthy     0.0981    0.4615    0.1618       156
        rest     0.9765    0.8405    0.9034      4150

    accuracy                     

/tmp/ipykernel_3038612/3346683839.py:253: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[efficientnet_b0] Epoch 003/20 | train_loss 0.1866 train_acc 0.8653 | val_loss 0.2646 val_acc 0.9065 | macro_f1 0.6859 | lr 5.42e-05
Per-class acc: 0(unhealthy)=94.87%  1(rest)=90.46%
Saved best model to: /home/ehr/Desktop/anupama/plant/training_runs/stage1_unhealthy_vs_rest_imagenet_only_no_aug_efficientnet_b0/best_model.pt
[efficientnet_b0] Epoch 004/20 | train_loss 0.0863 train_acc 0.9541 | val_loss 0.2994 val_acc 0.9384 | macro_f1 0.6405 | lr 1.56e-04
Per-class acc: 0(unhealthy)=36.54%  1(rest)=96.31%
[efficientnet_b0] Epoch 005/20 | train_loss 0.0625 train_acc 0.9715 | val_loss 0.1480 val_acc 0.9604 | macro_f1 0.8008 | lr 2.58e-04
Per-class acc: 0(unhealthy)=90.38%  1(rest)=96.24%
              precision    recall  f1-score   support

   unhealthy     0.4747    0.9038    0.6225       156
        rest     0.9963    0.9624    0.9790      4150

    accuracy                         0.9603      4306
   macro avg     0.7355    0.9331    0.8008      4306
weighted avg     0.9774    0.9603